# T36 - 同花顺Excel导入债券估值 重构

## 元数据
- 原始路径: g:\\我的云端硬盘\\workspace\\projects\\github\\work_station\\代码库\\模版\\同花顺excel导入债券估值等\\
- 创建时间: 2026-02-14
- 任务ID: T36
- 优先级: 最高
- 依赖: 同花顺iFinD、数据库、Python环境、Pandas、OpenPyXL、pywin32

## 1. 项目概述

### 1.1 功能描述

同花顺iFinD债券估值批量处理系统，主要功能包括：

1. **批量生成Excel模板文件**：从数据库查询日期+债券代码组合，生成Excel模板
2. **批量填充iFinD公式**：在Excel模板中批量插入同花顺iFinD公式
3. **批量计算并保存为值**：使用Excel COM接口执行公式计算，将结果保存为静态值

### 1.2 数据流

```
数据库查询 -> Excel模板生成 -> 添加iFinD公式 -> Excel计算 -> 保存为值 -> (待实现：回传数据库)
    ↓           ↓                ↓              ↓          ↓
  SQL       openpyxl         openpyxl      win32com    win32com
```

### 1.3 核心数据指标

| 指标名称 | 说明 | 列位置 |
|---------|------|-------|
| ths_bond_balance_bond | 债券余额 | C列 |
| ths_evaluate_yield_yy_bond | 银行间收益率 | D列 |
| ths_cb_market_implicit_rating_bond | 隐含评级 | E列 |
| ths_evaluate_yield_cb_bond_exercise | 交易所收益率 | F列 |
| ths_evaluate_modified_dur_cb_bond_exercise | 修正久期 | G列 |

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from functools import wraps

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import pymysql
import sqlalchemy
from sqlalchemy import text, inspect
from sqlalchemy.pool import NullPool

# Excel处理
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# Windows COM接口（仅Windows环境）
try:
    import win32com.client
    WIN32_AVAILABLE = True
except ImportError:
    WIN32_AVAILABLE = False
    print("警告: pywin32未安装，Excel COM功能不可用")

# 配置导入
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, TEMP_DIR,
    DB_CONNECTION_STRING, DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME,
    MAX_ROWS_PER_FILE, IFIND_WAIT_TIME, RETRY_ATTEMPTS, RETRY_DELAY,
    IFIND_INDICATORS, BOND_TYPES, YIELD_MIN, YIELD_MAX, REQUIRED_COLUMNS,
    validate_config
)

print("依赖导入成功")
print(f"Windows COM可用: {WIN32_AVAILABLE}")

### 2.2 数据库连接

In [ ]:
# 验证配置
if not validate_config():
    raise ValueError("配置验证失败，请检查环境变量设置")

# 创建数据库引擎
sql_engine = sqlalchemy.create_engine(
    DB_CONNECTION_STRING,
    poolclass=NullPool,
    pool_recycle=3600  # 每小时重连，防止连接超时
)

# 测试连接
try:
    with sql_engine.connect() as conn:
        result = conn.execute(text("SELECT 1")).fetchone()
        print(f"数据库连接成功: {DB_HOST}:{DB_PORT}/{DB_NAME}")
except Exception as e:
    print(f"数据库连接失败: {e}")
    raise

## 3. 数据库查询模块

### 3.1 存续债券查询逻辑

核心SQL逻辑说明：
- **起息日条件**：起息日 < 当前日期 OR 为空
- **到期日条件**：到期日 > 当前日期 OR 为空
- **退市日条件**：退市日 > 当前日期 OR 为空
- **发行失败排除**：排除发行失败的债券
- **去重逻辑**：排除已有估值数据的债券

In [ ]:
def build_survival_bond_query(bond_type, include_temp_filter=True):
    """
    构建查询存续期内债券的SQL语句
    
    参数:
    - bond_type: 债券类型 ('credit', 'finance', 'abs')
    - include_temp_filter: 是否包含临时表去重
    
    返回:
    - SQL查询字符串
    """
    if bond_type not in BOND_TYPES:
        raise ValueError(f"不支持的债券类型: {bond_type}")
    
    config = BOND_TYPES[bond_type]
    table_name = config['table']
    
    # 基础查询
    query = f"""
    SELECT 
        dt.dt as dt, 
        basicinfo.trade_code as trade_code
    FROM 
        temp_dt as dt
    JOIN
        {table_name} as basicinfo
    ON
        (
            -- 起息日条件：起息日 < 当前日期 OR 为空
            (basicinfo.ths_interest_begin_date_bond < dt.dt 
             OR basicinfo.ths_interest_begin_date_bond IS NULL 
             OR basicinfo.ths_interest_begin_date_bond = '0000-00-00')
            
            -- 到期日条件：到期日 > 当前日期 OR 为空
            AND (basicinfo.ths_maturity_date_bond > dt.dt 
                 OR basicinfo.ths_maturity_date_bond IS NULL 
                 OR basicinfo.ths_maturity_date_bond = '0000-00-00')
            
            -- 退市日条件：退市日 > 当前日期 OR 为空
            AND (basicinfo.ths_delist_date_bond > dt.dt 
                 OR basicinfo.ths_delist_date_bond IS NULL 
                 OR basicinfo.ths_delist_date_bond = '0000-00-00')
            
            -- 排除发行失败的债券
            AND basicinfo.ths_is_issuing_failure_bond != '是'
        )
    """
    
    # 添加去重条件
    if include_temp_filter and config.get('temp_table'):
        query += f"""
        AND basicinfo.TRADE_CODE NOT IN (
            SELECT trade_code FROM {config['temp_table']}
        )
        """
    
    return query


def build_union_query(bond_types=None, include_temp_filter=True):
    """
    构建多债券类型联合查询的SQL语句
    
    参数:
    - bond_types: 债券类型列表，默认为 ['credit', 'finance', 'abs']
    - include_temp_filter: 是否包含临时表去重
    
    返回:
    - SQL查询字符串
    """
    if bond_types is None:
        bond_types = list(BOND_TYPES.keys())
    
    queries = []
    for bond_type in bond_types:
        queries.append(build_survival_bond_query(bond_type, include_temp_filter))
    
    return "\nUNION ALL\n".join(queries)


print("SQL查询构建函数定义完成")

### 3.2 执行查询与生成Excel模板

In [ ]:
def query_dt_trade_code(bond_types=None, engine=None):
    """
    查询日期+债券代码组合
    
    参数:
    - bond_types: 债券类型列表
    - engine: SQLAlchemy引擎
    
    返回:
    - DataFrame包含dt和trade_code列
    """
    if engine is None:
        engine = sql_engine
    
    query = build_union_query(bond_types)
    
    print("执行查询...")
    print(f"查询债券类型: {bond_types or list(BOND_TYPES.keys())}")
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql_query(text(query), conn)
        print(f"查询完成，共 {len(df)} 条记录")
        return df
    except Exception as e:
        print(f"查询失败: {e}")
        raise


def generate_excel_templates(df, output_dir=None, max_rows=None):
    """
    将查询结果分割并保存为多个Excel文件
    
    参数:
    - df: 包含dt和trade_code的DataFrame
    - output_dir: 输出目录
    - max_rows: 每个文件的最大行数
    
    返回:
    - 生成的文件列表
    """
    if output_dir is None:
        output_dir = TEMP_DIR
    if max_rows is None:
        max_rows = MAX_ROWS_PER_FILE
    
    # 确保输出目录存在
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    total_rows = len(df)
    if total_rows == 0:
        print("没有数据需要导出")
        return []
    
    files = []
    num_files = (total_rows + max_rows - 1) // max_rows
    
    print(f"总记录数: {total_rows}")
    print(f"每文件最大行数: {max_rows}")
    print(f"预计生成文件数: {num_files}")
    
    for i in range(0, total_rows, max_rows):
        file_index = i // max_rows + 1
        file_path = Path(output_dir) / f"{file_index}.xlsx"
        
        chunk = df[i:i + max_rows]
        chunk.to_excel(file_path, index=False)
        
        files.append(str(file_path))
        print(f"生成文件: {file_path} ({len(chunk)} 行)")
    
    return files


print("Excel模板生成函数定义完成")

## 4. Excel公式填充模块

### 4.1 iFinD公式定义

iFinD公式格式：`=@thsiFinD("指标名", trade_code单元格, dt单元格, [参数])`

- A列：dt（日期）
- B列：trade_code（债券代码）
- C-G列：iFinD公式结果

In [ ]:
def build_ifind_formula(indicator_name, row, params=None):
    """
    构建iFinD公式字符串
    
    参数:
    - indicator_name: 指标名称
    - row: 行号
    - params: 可选参数（如102表示交易所）
    
    返回:
    - 公式字符串
    """
    if params is not None:
        return f'=@thsiFinD("{indicator_name}",B{row},A{row},{params})'
    else:
        return f'=@thsiFinD("{indicator_name}",B{row},A{row})'


def add_ifind_formulas_to_sheet(sheet, start_row=2, end_row=None):
    """
    向工作表批量添加iFinD公式
    
    参数:
    - sheet: openpyxl工作表对象
    - start_row: 开始行（默认第2行，跳过表头）
    - end_row: 结束行（默认为最大行）
    """
    if end_row is None:
        end_row = sheet.max_row
    
    # 添加表头
    for indicator_name, config in IFIND_INDICATORS.items():
        col = int(config['column'].replace('C', '3').replace('D', '4').replace('E', '5').replace('F', '6').replace('G', '7').replace('H', '8'))
        if config['column'] == 'C':
            col = 3
        elif config['column'] == 'D':
            col = 4
        elif config['column'] == 'E':
            col = 5
        elif config['column'] == 'F':
            col = 6
        elif config['column'] == 'G':
            col = 7
        
        sheet.cell(row=1, column=col).value = indicator_name
    
    # 批量添加公式
    for row in range(start_row, end_row + 1):
        for indicator_name, config in IFIND_INDICATORS.items():
            col = ord(config['column']) - ord('A') + 1
            formula = build_ifind_formula(indicator_name, row, config['params'])
            sheet.cell(row=row, column=col).value = formula
    
    print(f"添加公式完成: {end_row - start_row + 1} 行")


def add_formulas_to_excel(file_path):
    """
    向Excel文件添加iFinD公式
    
    参数:
    - file_path: Excel文件路径
    """
    try:
        book = load_workbook(file_path)
        sheet = book.active
        
        print(f"处理文件: {file_path}")
        print(f"数据行数: {sheet.max_row}")
        
        add_ifind_formulas_to_sheet(sheet)
        
        book.save(file_path)
        print(f"保存完成: {file_path}")
        
    except Exception as e:
        print(f"处理文件失败: {file_path}, 错误: {e}")
        raise


print("公式填充函数定义完成")

## 5. Excel计算与保存模块

### 5.1 重试装饰器

Excel COM接口不稳定，需要重试机制处理：
- 网络超时
- Excel崩溃
- iFinD API延迟

In [ ]:
def retry(attempts=None, delay=None):
    """
    重试装饰器
    
    参数:
    - attempts: 重试次数
    - delay: 重试间隔（秒）
    """
    if attempts is None:
        attempts = RETRY_ATTEMPTS
    if delay is None:
        delay = RETRY_DELAY
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"尝试 {i+1}/{attempts} 失败: {e}")
                    if i < attempts - 1:
                        print(f"{delay}秒后重试...")
                        time.sleep(delay)
            raise Exception(f"所有 {attempts} 次尝试均失败")
        return wrapper
    return decorator


print("重试装饰器定义完成")

### 5.2 Excel COM操作类

In [ ]:
class ExcelCOMApp:
    """
    Excel COM操作类
    封装win32com对Excel的操作，提供重试机制
    """
    
    def __init__(self):
        if not WIN32_AVAILABLE:
            raise RuntimeError("pywin32未安装，无法使用COM功能")
        
        self.Excel = win32com.client.Dispatch("Excel.Application")
        self.Excel.DisplayAlerts = False  # 关闭弹窗警告
        self.Excel.Visible = False  # 后台运行
        print("Excel COM实例创建成功")
    
    @retry()
    def open_workbook(self, file_path):
        """打开工作簿"""
        print(f"打开文件: {file_path}")
        return self.Excel.Workbooks.Open(file_path)
    
    @retry()
    def calculate_worksheet(self, ws):
        """执行工作表计算"""
        print("执行公式计算...")
        ws.Calculate()
    
    @retry()
    def save_as_values_copy_paste(self, ws):
        """
        将公式结果保存为值（Copy-PasteSpecial方式）
        性能较好，适合大数据量
        """
        print("保存为值（Copy-Paste方式）...")
        used_range = ws.UsedRange
        used_range.Copy()
        used_range.PasteSpecial(Paste=12)  # xlPasteValuesAndNumberFormats
        ws.Application.CutCopyMode = False
    
    @retry()
    def save_as_values_cell_by_cell(self, ws):
        """
        将公式结果保存为值（逐单元格方式）
        性能较差，但更可靠
        """
        print("保存为值（逐单元格方式）...")
        for cell in ws.UsedRange:
            cell.Value = cell.Value
    
    @retry()
    def save_workbook(self, wb):
        """保存工作簿"""
        print("保存文件...")
        wb.Save()
    
    @retry()
    def close_workbook(self, wb):
        """关闭工作簿"""
        print("关闭文件...")
        wb.Close()
    
    def quit(self):
        """退出Excel应用"""
        try:
            self.Excel.Quit()
            print("Excel应用已退出")
        except Exception as e:
            print(f"退出Excel时出错: {e}")


print("Excel COM类定义完成")

### 5.3 批量处理流程

In [ ]:
def process_excel_files_with_com(folder_path, wait_time=None, use_fast_save=True):
    """
    批量处理Excel文件：打开、计算、保存为值
    
    参数:
    - folder_path: 文件夹路径
    - wait_time: iFinD数据加载等待时间（秒）
    - use_fast_save: 是否使用快速保存方式（Copy-Paste）
    
    返回:
    - 处理结果列表
    """
    if wait_time is None:
        wait_time = IFIND_WAIT_TIME
    
    if not WIN32_AVAILABLE:
        print("错误: Windows COM不可用，无法执行")
        return []
    
    # 获取所有xlsx文件
    folder = Path(folder_path)
    files = sorted([f for f in folder.glob("*.xlsx")])
    
    if not files:
        print(f"文件夹 {folder_path} 中没有xlsx文件")
        return []
    
    print(f"找到 {len(files)} 个文件待处理")
    
    results = []
    excel = ExcelCOMApp()
    
    try:
        for i, file_path in enumerate(files):
            print(f"\n{'='*50}")
            print(f"处理文件 [{i+1}/{len(files)}]: {file_path.name}")
            print(f"{'='*50}")
            
            try:
                # Step 1: 打开Excel
                wb = excel.open_workbook(str(file_path.absolute()))
                
                # Step 2: 等待iFinD数据加载
                print(f"等待iFinD数据加载 ({wait_time}秒)...")
                time.sleep(wait_time)
                
                # Step 3: 执行公式计算
                excel.calculate_worksheet(wb.Worksheets(1))
                
                # Step 4: 保存为值
                if use_fast_save:
                    excel.save_as_values_copy_paste(wb.Worksheets(1))
                else:
                    excel.save_as_values_cell_by_cell(wb.Worksheets(1))
                
                # Step 5: 保存文件
                excel.save_workbook(wb)
                
                # Step 6: 关闭文件
                excel.close_workbook(wb)
                
                results.append({
                    'file': file_path.name,
                    'status': 'success'
                })
                print(f"文件处理完成: {file_path.name}")
                
            except Exception as e:
                results.append({
                    'file': file_path.name,
                    'status': 'failed',
                    'error': str(e)
                })
                print(f"文件处理失败: {file_path.name}, 错误: {e}")
    
    finally:
        excel.quit()
    
    # 打印汇总
    success_count = sum(1 for r in results if r['status'] == 'success')
    failed_count = len(results) - success_count
    print(f"\n{'='*50}")
    print(f"批量处理完成")
    print(f"成功: {success_count}, 失败: {failed_count}")
    print(f"{'='*50}")
    
    return results


print("批量处理函数定义完成")

## 6. 数据验证模块

### 6.1 数据完整性验证

In [ ]:
def validate_excel_data(file_path):
    """
    验证Excel数据完整性
    
    参数:
    - file_path: Excel文件路径
    
    返回:
    - (is_valid, issues)
    """
    issues = []
    
    try:
        df = pd.read_excel(file_path)
        
        # 1. 检查必需列
        missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
        if missing_cols:
            issues.append(f"缺少必需列: {missing_cols}")
        
        # 2. 检查空值
        null_counts = df.isnull().sum()
        for col, count in null_counts.items():
            if count > 0:
                issues.append(f"列 '{col}' 有 {count} 个空值")
        
        # 3. 检查估值收益率范围
        if 'ths_evaluate_yield_yy_bond' in df.columns:
            yield_col = pd.to_numeric(df['ths_evaluate_yield_yy_bond'], errors='coerce')
            invalid_yield = yield_col[(yield_col < YIELD_MIN) | (yield_col > YIELD_MAX)]
            if len(invalid_yield) > 0:
                issues.append(f"估值收益率超出范围 [{YIELD_MIN}, {YIELD_MAX}]: {len(invalid_yield)} 条")
        
        # 4. 检查日期格式
        if 'dt' in df.columns:
            try:
                pd.to_datetime(df['dt'])
            except Exception:
                issues.append("日期列格式不正确")
        
        is_valid = len(issues) == 0
        return is_valid, issues
        
    except Exception as e:
        return False, [f"读取文件失败: {e}"]


def validate_batch(folder_path):
    """
    批量验证文件夹中的Excel文件
    
    参数:
    - folder_path: 文件夹路径
    
    返回:
    - 验证结果DataFrame
    """
    folder = Path(folder_path)
    files = sorted(folder.glob("*.xlsx"))
    
    results = []
    for file_path in files:
        is_valid, issues = validate_excel_data(file_path)
        results.append({
            'file': file_path.name,
            'is_valid': is_valid,
            'issues': '; '.join(issues) if issues else ''
        })
    
    df = pd.DataFrame(results)
    
    print(f"\n验证汇总:")
    print(f"总文件数: {len(df)}")
    print(f"验证通过: {df['is_valid'].sum()}")
    print(f"验证失败: {(~df['is_valid']).sum()}")
    
    if (~df['is_valid']).sum() > 0:
        print("\n失败详情:")
        for _, row in df[~df['is_valid']].iterrows():
            print(f"  - {row['file']}: {row['issues']}")
    
    return df


print("数据验证函数定义完成")

## 7. 主流程整合

### 7.1 完整工作流

In [ ]:
def run_full_workflow(bond_types=None, output_dir=None, execute_com=True):
    """
    执行完整工作流
    
    参数:
    - bond_types: 债券类型列表
    - output_dir: 输出目录
    - execute_com: 是否执行COM计算（需要Windows + Excel）
    
    返回:
    - 工作流结果
    """
    print("="*60)
    print("T36 - 同花顺Excel导入债券估值 工作流开始")
    print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    if output_dir is None:
        output_dir = OUTPUT_DIR
    
    workflow_results = {
        'start_time': datetime.now(),
        'steps': {}
    }
    
    # Step 1: 查询数据库
    print("\n[Step 1] 查询数据库...")
    try:
        df = query_dt_trade_code(bond_types)
        workflow_results['steps']['query'] = {
            'status': 'success',
            'record_count': len(df)
        }
    except Exception as e:
        workflow_results['steps']['query'] = {
            'status': 'failed',
            'error': str(e)
        }
        print(f"查询失败: {e}")
        return workflow_results
    
    # Step 2: 生成Excel模板
    print("\n[Step 2] 生成Excel模板...")
    try:
        files = generate_excel_templates(df, output_dir)
        workflow_results['steps']['generate_templates'] = {
            'status': 'success',
            'file_count': len(files),
            'files': files
        }
    except Exception as e:
        workflow_results['steps']['generate_templates'] = {
            'status': 'failed',
            'error': str(e)
        }
        print(f"生成模板失败: {e}")
        return workflow_results
    
    # Step 3: 添加iFinD公式
    print("\n[Step 3] 添加iFinD公式...")
    try:
        for file_path in files:
            add_formulas_to_excel(file_path)
        workflow_results['steps']['add_formulas'] = {
            'status': 'success',
            'processed_files': len(files)
        }
    except Exception as e:
        workflow_results['steps']['add_formulas'] = {
            'status': 'failed',
            'error': str(e)
        }
        print(f"添加公式失败: {e}")
        return workflow_results
    
    # Step 4: 执行COM计算（可选）
    if execute_com and WIN32_AVAILABLE:
        print("\n[Step 4] 执行Excel计算并保存为值...")
        try:
            com_results = process_excel_files_with_com(output_dir)
            workflow_results['steps']['com_calculate'] = {
                'status': 'success',
                'results': com_results
            }
        except Exception as e:
            workflow_results['steps']['com_calculate'] = {
                'status': 'failed',
                'error': str(e)
            }
            print(f"COM计算失败: {e}")
    else:
        print("\n[Step 4] 跳过COM计算（需要Windows环境或手动执行）")
        workflow_results['steps']['com_calculate'] = {
            'status': 'skipped',
            'reason': 'Windows COM not available' if not WIN32_AVAILABLE else 'Disabled by user'
        }
    
    # Step 5: 数据验证
    print("\n[Step 5] 数据验证...")
    try:
        validation_df = validate_batch(output_dir)
        workflow_results['steps']['validation'] = {
            'status': 'success',
            'valid_count': validation_df['is_valid'].sum(),
            'invalid_count': (~validation_df['is_valid']).sum()
        }
    except Exception as e:
        workflow_results['steps']['validation'] = {
            'status': 'failed',
            'error': str(e)
        }
        print(f"验证失败: {e}")
    
    workflow_results['end_time'] = datetime.now()
    duration = (workflow_results['end_time'] - workflow_results['start_time']).total_seconds()
    
    print("\n" + "="*60)
    print("工作流完成")
    print(f"耗时: {duration:.2f} 秒")
    print("="*60)
    
    return workflow_results


print("主流程函数定义完成")

### 7.2 分步执行模式

如果需要手动控制每个步骤，可以使用以下独立函数：

In [ ]:
def step1_query_data(bond_types=None):
    """
    Step 1: 查询数据库
    
    返回:
    - DataFrame
    """
    print("Step 1: 查询数据库")
    df = query_dt_trade_code(bond_types)
    print(f"查询完成，共 {len(df)} 条记录")
    return df


def step2_generate_templates(df, output_dir=None):
    """
    Step 2: 生成Excel模板
    
    参数:
    - df: 数据DataFrame
    - output_dir: 输出目录
    
    返回:
    - 文件路径列表
    """
    print("Step 2: 生成Excel模板")
    files = generate_excel_templates(df, output_dir)
    print(f"生成完成，共 {len(files)} 个文件")
    return files


def step3_add_formulas(files):
    """
    Step 3: 添加iFinD公式
    
    参数:
    - files: 文件路径列表
    """
    print("Step 3: 添加iFinD公式")
    for file_path in files:
        add_formulas_to_excel(file_path)
    print(f"公式添加完成")


def step4_calculate_and_save(folder_path):
    """
    Step 4: 执行计算并保存为值
    
    参数:
    - folder_path: 文件夹路径
    
    返回:
    - 处理结果列表
    """
    print("Step 4: 执行计算并保存为值")
    if not WIN32_AVAILABLE:
        print("警告: Windows COM不可用")
        return []
    return process_excel_files_with_com(folder_path)


def step5_validate(folder_path):
    """
    Step 5: 数据验证
    
    参数:
    - folder_path: 文件夹路径
    
    返回:
    - 验证结果DataFrame
    """
    print("Step 5: 数据验证")
    return validate_batch(folder_path)


print("分步执行函数定义完成")

---

## 使用说明

### 环境要求

1. **Python环境**: Python 3.8+
2. **数据库**: MySQL数据库，包含债券基础信息表
3. **同花顺iFinD**: 需要安装同花顺iFinD客户端和Excel插件
4. **Windows环境**: COM功能仅Windows可用

### 环境变量配置

```bash
# 数据库配置
export DB_HOST=localhost
export DB_PORT=3306
export DB_USER=your_username
export DB_PASSWORD=your_password
export DB_NAME=bond
```

### 执行方式

```python
# 完整工作流
results = run_full_workflow()

# 或分步执行
df = step1_query_data()
files = step2_generate_templates(df)
step3_add_formulas(files)
step4_calculate_and_save(OUTPUT_DIR)
step5_validate(OUTPUT_DIR)
```

---

**最后更新**: 2026-02-14